> **Historical record.** These questions were asked of an earlier version of the model
> (108 inputs, 3-fight window). The input review and its end tests that followed superseded
> them: the model now has 40 chosen inputs and an 8-fight window (`reviews/`). This notebook is kept as
> it ran and is **not re-run**; its code would no longer reproduce these numbers.

# 07 — The feature questions, on the tuned model

Individual features were not judged until the model was tuned: an
untuned model is a bad judge. Tuning has happened, so these get their fair hearing now,
together with one new feature (stance as a matchup):

| Version | What changes | The question |
|---|---|---|
| `no_shrinkage` | ratio features unshrunk (alpha = 0); 0/0 stays blank | Does shrinkage help? |
| `no_weight_class` | without `weight_class_lbs` and `is_womens_bout` | Do the weight-class features help? |
| `mens_only` | trained on men's fights only | Pooled or men-only training? |
| `without_weak_groups` | without `record`, `pace`, `damage`, `cold_start` (14 columns) | Do the four weakest groups (per SHAP) help? |
| `stance` | adds `diff_static_southpaw` and `diff_static_switch` | Does stance as a matchup help? |

Each version is one change from the current model (`baseline`), and each gets **its own
tuned settings**: the same search that chose the current ones (60 trials, 3 seeds), so no
version is judged with settings chosen for another.

**The rule, fixed before this run** (the same one the Elo decision used in `04_elo.ipynb`).
The current model stays unless a version beats it on both:

1. **Cross-validation** inside the training years (2019–2022 scored, 8 seeds): lower log
   loss in **at least 7 of 8 seeds**.
2. **Validation** (2023–24, men's fights): log loss **not worse**.

`mens_only` can only predict men's fights, so pooled vs men-only is compared on men's
fights in both steps; every other version is compared on all fights in cross-validation.

Next to each result, the **paired log-loss test** (`evaluate.compare_log_loss`): the
average per-fight difference, and how big that is compared with the noise from which
fights happened to be scored. It doesn't decide; it says how much weight a decision
deserves. Nothing here saves any file: if a version wins, adopting it is a separate step.

In [1]:
import time

import numpy as np
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)

from ufc_predictor.data.io import load_fighters, load_processed
from ufc_predictor.features import manifest
from ufc_predictor.features.elo import load_elo_params
from ufc_predictor.features.pipeline import build_featured_table
from ufc_predictor.features.rolling import no_shrinkage
from ufc_predictor.features.static import attach_stance_features
from ufc_predictor.modeling.evaluate import compare, compare_log_loss, is_womens_fight, per_fight_log_loss, score
from ufc_predictor.modeling.model import fit_models, predict_proba_mean
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import cv_predictions, load_params, seed_columns, tune

N_TRIALS = 60       # the same search as the current settings
SEEDS = range(8)      # model seeds for every comparison

fights, fighters = load_processed("fights"), load_fighters()
featured = load_processed("fights_featured")
train, validation, _ = chronological_split(featured)      # the test years stay closed

WEAK_GROUPS = ("record", "pace", "damage", "cold_start")
weak_columns = [c for g in WEAK_GROUPS for c in manifest.feature_groups(train)[g]]

tables = {
    "baseline": featured,
    "no_shrinkage": build_featured_table(fights, fighters, no_shrinkage(), elo_k=load_elo_params()),
    "no_weight_class": featured.drop(columns=["weight_class_lbs", "is_womens_bout"]),
    "mens_only": featured,                       # same table, fewer training rows (below)
    "without_weak_groups": featured.drop(columns=weak_columns),
    "stance": attach_stance_features(featured),
}

def mens(table):
    return table[~is_womens_fight(table, history=featured).to_numpy()].reset_index(drop=True)

splits = {}
for name, table in tables.items():
    tr, va, _ = chronological_split(table)
    assert (va["fights_url"].to_numpy() == validation["fights_url"].to_numpy()).all()
    splits[name] = (mens(tr), mens(va)) if name == "mens_only" else (tr, va)

pd.DataFrame([
    {"version": name, "features": len(manifest.feature_columns(tr)), "training_fights": len(tr)}
    for name, (tr, _) in splits.items()
]).set_index("version")

,features,training_fights
version,,
baseline,108,4309
no_shrinkage,108,4309
no_weight_class,106,4309
mens_only,108,3692
without_weak_groups,94,4309
stance,110,4309


## 1. Tune each version

The current settings (`model_params.json`) came from exactly this search on the current
table, so `baseline` reuses them. Every other version gets its own search. About 10–15
minutes per version.

In [2]:
params = {"baseline": load_params()}
best_cv = {}
for name, (tr, _) in splits.items():
    if name == "baseline":
        continue
    start = time.time()
    study = tune(tr, n_trials=N_TRIALS)
    params[name], best_cv[name] = study.best_params, study.best_value
    print(f"{name:20s} best CV {study.best_value:.5f}  ({(time.time() - start) / 60:.0f} min)")

pd.DataFrame(params).T.round(4)

no_shrinkage         best CV 0.66081  (22 min)


no_weight_class      best CV 0.66247  (12 min)


mens_only            best CV 0.65963  (8 min)


without_weak_groups  best CV 0.66557  (12 min)


stance               best CV 0.66319  (10 min)


,l2_regularization,learning_rate,max_features,max_iter,max_leaf_nodes,min_samples_leaf
baseline,23.3031,0.0173,0.3343,400.0,10.0,18.0
no_shrinkage,26.7772,0.0106,0.5107,550.0,13.0,7.0
no_weight_class,15.9148,0.0147,0.3485,400.0,11.0,6.0
mens_only,24.6335,0.0328,0.3805,200.0,9.0,18.0
without_weak_groups,0.1795,0.0109,0.4345,450.0,9.0,5.0
stance,16.5493,0.0246,0.5355,250.0,7.0,5.0


## 2. Cross-validation, 8 seeds

Each version's own settings, the same folds (train on earlier years, score 2019, 2020,
2021 and 2022 in turn) and the same 8 model seeds. `better_seeds` counts the seeds in
which the version beat the current model. The paired test uses the predictions averaged
over the 8 seeds, matched fight by fight.

In [3]:
cv = {name: cv_predictions(tr, params[name], seeds=SEEDS) for name, (tr, _) in splits.items()}

def seed_scores(predictions):
    return np.array([per_fight_log_loss(predictions["fighter_a_won"], predictions[c]).mean()
                     for c in seed_columns(predictions)])

def averaged(predictions):
    return predictions[seed_columns(predictions)].mean(axis=1).to_numpy()

rows = []
for name, predictions in cv.items():
    if name == "baseline":
        continue
    # line the current model's predictions up with this version's scored fights
    base = cv["baseline"].set_index("fights_url").loc[predictions["fights_url"]].reset_index()
    new, old = seed_scores(predictions), seed_scores(base)
    paired = compare_log_loss(predictions["fighter_a_won"], averaged(predictions), averaged(base))
    rows.append({
        "version": name, "scored_on": "men's fights" if name == "mens_only" else "all fights",
        "fights": len(predictions), "log_loss": new.mean(), "baseline_log_loss": old.mean(),
        "better_seeds": f"{(new < old).sum()} of {len(new)}",
        "difference": paired["difference"], "se": paired["se"], "p_value": paired["p_value"],
    })
cv_table = pd.DataFrame(rows).set_index("version")
cv_table.round(5)

,scored_on,fights,log_loss,baseline_log_loss,better_seeds,difference,se,p_value
version,,,,,,,,
no_shrinkage,all fights,1953,0.66061,0.66258,8 of 8,-0.00191,0.00212,0.36678
no_weight_class,all fights,1953,0.66259,0.66258,6 of 8,0.00002,0.00030,0.95510
mens_only,men's fights,1596,0.66023,0.65851,0 of 8,0.00160,0.00129,0.21697
without_weak_groups,all fights,1953,0.66567,0.66258,0 of 8,0.00310,0.00168,0.06555
stance,all fights,1953,0.66326,0.66258,1 of 8,0.00069,0.00066,0.29525


## 3. One look at validation

Each version's final predictor: its own settings, 8 seeds, each fight asked both ways
round, trained on the training years and scored on 2023–24. Men's fights are the
headline, as always. `fixes` / `breaks` count men's fights whose pick changed for the
better / worse against the current model.

In [4]:
predicted = {}
for name, (tr, va) in splits.items():
    X_train, y_train = manifest.make_xy(tr)
    X_val, y_val = manifest.make_xy(va)
    predicted[name] = pd.DataFrame({
        "fights_url": va["fights_url"].to_numpy(),
        "fighter_a_won": np.asarray(y_val, dtype=bool),
        "p": predict_proba_mean(fit_models(X_train, y_train, params[name], seeds=SEEDS), X_val),
    })

mens_urls = set(mens(validation)["fights_url"])
rows = []
for name, frame in predicted.items():
    men = frame[frame["fights_url"].isin(mens_urls)]
    base = predicted["baseline"].set_index("fights_url").loc[men["fights_url"]].reset_index()
    s = score(men["fighter_a_won"], men["p"])
    row = {"version": name, "men_fights": s["fights"], "men_accuracy": s["accuracy"],
           "men_log_loss": s["log_loss"], "men_auc": s["auc"]}
    if name != "baseline":
        paired = compare_log_loss(men["fighter_a_won"], men["p"], base["p"])
        picks = compare(men["fighter_a_won"], men["p"], base["p"])
        row |= {"difference": paired["difference"], "p_value": paired["p_value"],
                "fixes": picks["fixes"], "breaks": picks["breaks"]}
    rows.append(row)
validation_table = pd.DataFrame(rows).set_index("version")
validation_table.round(4)

,men_fights,men_accuracy,men_log_loss,men_auc,difference,p_value,fixes,breaks
version,,,,,,,,
baseline,837,0.6213,0.6501,0.6731,NaN,NaN,NaN,NaN
no_shrinkage,837,0.6093,0.6552,0.6592,0.0050,0.0999,33.0,43.0
no_weight_class,837,0.6201,0.6499,0.6748,-0.0002,0.6211,4.0,5.0
mens_only,837,0.6093,0.6522,0.6667,0.0021,0.2140,19.0,29.0
without_weak_groups,837,0.6237,0.6506,0.6792,0.0005,0.8491,32.0,30.0
stance,837,0.6272,0.6494,0.6772,-0.0007,0.4681,13.0,8.0


## 4. The decisions

The rule from the top, applied as written.

In [5]:
base_val = validation_table.loc["baseline", "men_log_loss"]
decisions = []
for name in cv_table.index:
    wins, of = (int(x) for x in cv_table.loc[name, "better_seeds"].split(" of "))
    cv_pass = wins >= 7 * of / 8
    val_pass = validation_table.loc[name, "men_log_loss"] <= base_val
    decisions.append({
        "version": name,
        "cv_better_seeds": cv_table.loc[name, "better_seeds"],
        "cv_passes": cv_pass,
        "validation_change": validation_table.loc[name, "men_log_loss"] - base_val,
        "validation_passes": val_pass,
        "decision": "ADOPT" if cv_pass and val_pass else "keep the current model",
    })
pd.DataFrame(decisions).set_index("version").round(4)

,cv_better_seeds,cv_passes,validation_change,validation_passes,decision
version,,,,,
no_shrinkage,8 of 8,True,0.0050,False,keep the current model
no_weight_class,6 of 8,False,-0.0002,True,keep the current model
mens_only,0 of 8,False,0.0021,False,keep the current model
without_weak_groups,0 of 8,False,0.0005,False,keep the current model
stance,1 of 8,False,-0.0007,True,keep the current model


## What it shows

**None of the five changes passes the rule, so the current model stays exactly as it is.**
Question by question:

1. **Shrinkage: the one close call, and the two tests disagree.** Without shrinkage,
   cross-validation was better in all 8 seeds, but by only 0.0019, and the paired test
   says that is well inside the noise (p = 0.37). On validation it went the other way:
   0.0050 worse on men's fights (p = 0.10), with accuracy 60.9% against 62.1%. Neither
   result is convincing on its own. Validation contradicts cross-validation, so the rule
   keeps shrinkage, which also keeps its practical benefits: no 0/0 blanks, and no
   extreme values from tiny samples.
2. **Weight class: no difference at all.** A tie in cross-validation (+0.00002, 6 of 8
   seeds) and on validation (−0.0002; the two versions disagree on just 9 fights). It
   stays, on the MMA reasoning that a heavyweight and a flyweight fight are different.
3. **Pooled training is confirmed.** Training on men's fights only was worse in all 8
   seeds (+0.0016) and on validation (+0.0021). The extra women's fights help the model
   learn patterns that hold for everyone, now confirmed with each version tuned.
4. **The "weakest" groups earn their place.** Removing record, pace, damage and cold start
   was worse in all 8 seeds (+0.0031, p = 0.07). SHAP said they move predictions only a
   little; this says that little is still useful. Small is not the same as useless.
5. **Stance as a matchup adds nothing measurable.** Slightly worse in cross-validation
   (+0.0007, better in only 1 of 8 seeds) and slightly better on validation (−0.0007).
   Both are noise-sized, which matches the raw data: southpaws beat orthodox fighters
   52.2% of the time, inside the ±2.8% coin-flip range. It is not added.

A useful by-product: the tuned settings vary a lot between versions (L2 from 0.18 to 27)
while their scores barely move, which says the search landscape is flat. Many settings
are about equally good, so no single one should be read as "the" answer.